#STURM-Flood

In [1]:
#@title Imports

import random
import os
import sys
import zipfile

from dataclasses import dataclass
from pathlib import Path

In [2]:
#@title Mount Goole Drive

root_path = "/content/drive/MyDrive/MSc/Flood-Mapping"  #@param {type:"string", multiline:true}

Dataset_url = "https://huggingface.co/datasets/tax2310/STURM-fusion-24/resolve/main/Dataset.zip"  #@param {type:"string", multiline:true}

mount_drive = True  #@param {type:"boolean"}

clone_repo = False  #@param {type:"boolean"}

if clone_repo and mount_drive:

    from google.colab import drive
    drive.mount("/content/drive")

    root_path = os.path.join(root_path, "Flood-Mapping")

    !git clone https://github.com/TAX2310/Flood-Mapping.git $root_path

    sys.path.append(os.path.join(root_path))

    from src.config import CFG
    cfg = CFG()

    cfg.ROOT = Path(root_path)

elif not clone_repo and mount_drive :
    from google.colab import drive
    drive.mount("/content/drive")

    sys.path.append(os.path.join(root_path))

    from src.config import CFG
    cfg = CFG()

    cfg.ROOT = Path(root_path)

elif clone_repo and not mount_drive:
    root_path = "Flood-Mapping"

    !git clone https://github.com/TAX2310/Flood-Mapping.git

    sys.path.append(root_path)

    from config import CFG
    cfg = CFG()

    cfg.ROOT = Path(root_path)

else:
    from src.config import CFG
    cfg = CFG()

    sys.path.append(os.path.join(cfg.ROOT))

cfg.ZIP_URL = Dataset_url

Mounted at /content/drive


In [3]:
import src.data.sturm_fusion as SturmFusion


In [4]:
import importlib

importlib.reload(SturmFusion)
#cfg = CFG()
#sys.path.append(os.path.join(cfg.ROOT))

<module 'src.data.sturm_fusion' from '/content/drive/MyDrive/MSc/Flood-Mapping/src/data/sturm_fusion.py'>

In [5]:
#@title Download Dataset

data_root = SturmFusion.download_and_extract(cfg)

img_dir = cfg.DATA_PATH / cfg.S1_PATH
mask_dir = cfg.DATA_PATH / cfg.MASK_PATH

print(mask_dir)

print("Image dir exists:", img_dir.exists())
print("Mask dir exists:", mask_dir.exists())

print("Num images:", len(list(img_dir.glob("*.tif"))))
print("Num masks:", len(list(mask_dir.glob("*.tif"))))


⬇️ Downloading dataset...
📦 Extracting dataset...
Extracting to: /content/drive/MyDrive/MSc/Flood-Mapping
✅ Extraction complete.
🗑️ Zip file deleted.

📂 Final structure:
 - S1
 - S2
 - floodmaps
 - metadata
/content/drive/MyDrive/MSc/Flood-Mapping/Dataset/floodmaps
Image dir exists: True
Mask dir exists: True
Num images: 1969
Num masks: 1970


In [6]:
print(cfg.ROOT)
print(os.listdir(cfg.ROOT))
print(cfg.DATA_PATH)
print(os.listdir(cfg.DATA_PATH))
print(type(cfg.ZIP_PATH))

/content/drive/MyDrive/MSc/Flood-Mapping
['__pycache__', '.git', 'README.md', 'notes.txt', '.gitignore', 'src', 'requirements.txt', 'experiments', '01_setup.ipynb', '03_train_s2.ipynb', 'Copy of 03_train_s2.ipynb', 'Copy of 02_train_s1.ipynb', '02_train_s1.ipynb', 'Dataset']
/content/drive/MyDrive/MSc/Flood-Mapping/Dataset
['S1', 'S2', 'floodmaps', 'metadata']
<class 'pathlib.PosixPath'>


In [7]:
import src.util.seed as seed
import src.data.split as split
import src.data.dataset as dataset

seed.set_seed(cfg.RANDOM_SEED)

samples = split.build_s2_index(cfg)
if cfg.SPLIT_METHOD == "by_event":
    train_samples, val_samples, test_samples = split.split_by_event(samples, cfg)
elif cfg.SPLIT_METHOD == "random":
    train_samples, val_samples, test_samples = split.split_random(samples, cfg)

print(f"Train samples: {len(train_samples)}")
print(f"Val samples: {len(val_samples)}")
print(f"Test samples: {len(test_samples)}")

Train samples: 1575
Val samples: 196
Test samples: 198


In [8]:
import src.data.dataloader as dataloader

cfg.USE_ROTATIONS = False
cfg.BATCH_SIZE = 64
cfg.SHUFFLE_TRAIN = True
cfg.NUM_WORKERS = 0
cfg.PIN_MEMORY = False

train_loader, val_loader, _ = dataloader.make_s2_dataloaders(cfg)

print(f"Train loader: {len(train_loader)}")
print(f"Val loader: {len(val_loader)}")
#

Train loader: 25
Val loader: 4
